# TCN subsample run — evaluation notebook

Evaluate the prior: `bash run_tcn_baseline_160_original_instancenorm_subsample.sh --prebuilt-mmap-dir ./subseqs_original_mmap --batch-size 16`

1. Load **history.json** and plot train/val loss and val F1.
2. Load checkpoint and **validation data** (prebuilt mmap, decimate 10).
3. Visualize validation samples: **overlay GT disruption** and **predicted disruption** time steps on the signal.

**GT disruption** = first time step where the label is 1. In the pipeline this is **t_disruption − 300 ms** (Twarn), i.e. the *warning* time, not the actual disruption time. The target is built so label=1 from that point onward.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

# Paths — set to your run. On server use e.g.:
#   Temporary/dpark1/scratch/soen_fusion_zero/checkpoints_tcn_ddp_original/L4_H80_20260311_095201
CHECKPOINT_DIR = Path("checkpoints_tcn_ddp_original/L4_H80_20260311_095201")
PREBUILT_MMAP_DIR = "./subseqs_original_mmap"
DECIMATE_FACTOR = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Load and plot training history

In [ ]:
history_path = CHECKPOINT_DIR / "history.json"
if not history_path.exists():
    raise FileNotFoundError(f"Not found: {history_path}")

with open(history_path) as f:
    history = json.load(f)

epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].plot(epochs, history["train_loss"], label="Train")
axes[0, 0].plot(epochs, history["val_loss"], label="Val")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs, history["val_f1"], color="C2")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("F1")
axes[0, 1].set_title("Val F1")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs, history["val_acc"], label="Val acc@th")
axes[1, 0].plot(epochs, history["val_precision"], label="Precision")
axes[1, 0].plot(epochs, history["val_recall"], label="Recall")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Metric")
axes[1, 0].set_title("Val accuracy / P / R")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(epochs, history["val_threshold"], color="C4")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Threshold")
axes[1, 1].set_title("Val best threshold")
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle("TCN subsample run — training history", fontsize=12)
plt.tight_layout()
plt.show()

## 2. Load model and validation dataset

In [ ]:
# Import model builder and dataset (same as train_tcn_ddp_original)
from disruptcnn.dataset_original import PrebuiltOriginalSubseqDataset
from train_tcn_ddp_original import build_model

ckpt_path = CHECKPOINT_DIR / "best.pt"
if not ckpt_path.exists():
    ckpt_path = CHECKPOINT_DIR / "last.pt"
state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

args = state["args"]
nrecept = state["nrecept"]

model, _, _ = build_model(
    args["input_channels"], 1, args["levels"], args["nhid"],
    args["kernel_size"], args["dilation_base"], args["dropout"],
    nrecept_target=args["nrecept_target"],
    use_instance_norm=args.get("use_instance_norm", False),
    use_prenorm=args.get("use_prenorm", False),
)
model.load_state_dict(state["state_dict"])
model = model.to(DEVICE)
model.eval()

ds = PrebuiltOriginalSubseqDataset(PREBUILT_MMAP_DIR, decimate_factor=DECIMATE_FACTOR)
val_inds = ds.get_split_indices("val")
if len(val_inds) == 0:
    val_inds = ds.get_split_indices("test")

print(f"Checkpoint: {ckpt_path}")
print(f"nrecept: {nrecept}, val samples: {len(val_inds)}")

## 3. Run inference and get GT / predicted disruption timesteps

In [ ]:
def first_disrupt_step(target: np.ndarray) -> int:
    """First time step index where target >= 0.5. Return -1 if no disruption."""
    pos = np.where(np.asarray(target).ravel() >= 0.5)[0]
    return int(pos[0]) if len(pos) > 0 else -1


def predict_disrupt_step(model, x: torch.Tensor, nrecept: int, device: torch.device, thresh: float = 0.5) -> int:
    """
    x: (1, C, T). Returns first *sequence* time step where pred >= thresh.
    Model output is valid from index nrecept-1; pred index 0 -> sequence step nrecept-1.
    """
    with torch.no_grad():
        out = model(x)
    # out (1, T) from TCN; valid predictions from index nrecept-1
    pred = out[0, nrecept - 1:].cpu().numpy()
    pos = np.where(pred >= thresh)[0]
    if len(pos) == 0:
        return -1
    return int(nrecept - 1 + pos[0])


results = []
n_vis = min(12, len(val_inds))
indices_to_plot = np.linspace(0, len(val_inds) - 1, n_vis, dtype=int)

for idx in indices_to_plot:
    i = val_inds[idx]
    x, target, weight = ds[i]
    x = x.unsqueeze(0).to(DEVICE)
    # Flatten channel dims: (1, C, T) or (1, 20, 8, T) -> (1, 160, T); TCN expects (B, C, T)
    x = x.view(1, -1, x.shape[-1])
    gt_step = first_disrupt_step(target.numpy())
    pred_step = predict_disrupt_step(model, x, nrecept, DEVICE)
    with torch.no_grad():
        out = model(x)
    pred_dense = out[0, nrecept - 1:].cpu().numpy()  # (T_valid,) probability per step
    results.append({
        "val_idx": idx,
        "seq_idx": i,
        "x": x.cpu(),
        "target": target,
        "gt_step": gt_step,
        "pred_step": pred_step,
        "pred_dense": pred_dense,
    })

print(f"Computed GT and predicted disruption steps for {len(results)} validation samples.")

## 4. Visualize: signal with overlaid GT and predicted disruption time steps

In [ ]:
def plot_signal_gt_pred(r, ax, signal="mean"):
    """
    Plot one validation sample: signal vs time, vertical lines at GT and predicted disruption.
    r: dict with x (1,C,T), target, gt_step, pred_step.
    signal: 'mean' (mean over channels) or 'first' (first channel).
    """
    x = r["x"][0].numpy()  # (C, T)
    if signal == "mean":
        sig = x.mean(axis=0)
    else:
        sig = x[0]
    T = sig.shape[0]
    ax.plot(np.arange(T), sig, color="#333", linewidth=0.6, alpha=0.9, label="Signal (mean ch)")
    gt = r["gt_step"]
    pred = r["pred_step"]
    if gt >= 0:
        ax.axvline(gt, color="green", linestyle="-", linewidth=1.5, label="GT disruption")
    if pred >= 0:
        ax.axvline(pred, color="red", linestyle="--", linewidth=1.5, label="Pred disruption")
    ax.set_xlabel("Time (decimated step)")
    ax.set_ylabel("Amplitude")
    ax.legend(loc="upper right", fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_title(f"Val idx {r['val_idx']} (seq {r['seq_idx']})")


n_cols = 3
n_rows = (n_vis + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)
for k, r in enumerate(results):
    i, j = k // n_cols, k % n_cols
    plot_signal_gt_pred(r, axes[i, j])
for k in range(len(results), n_rows * n_cols):
    i, j = k // n_cols, k % n_cols
    axes[i, j].set_visible(False)

plt.suptitle("Validation samples: signal with GT (green) and predicted (red) disruption time step", fontsize=11)
plt.tight_layout()
plt.savefig("evaluation_subsample_gt_vs_pred.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Dense labeling: GT and predicted probability over time (where mistakes lie)

Plot the **full per-timestep label** (GT) and **model probability** (pred_dense), not just the first-hit bar. This shows where the model wrongly fires on **clear** subsequences (false positives) or misses on disruptive ones (false negatives).

In [ ]:
THRESH = 0.5

def plot_dense_labels(r, ax, nrecept, thresh=0.5):
    """
    Plot dense GT label (0/1) and dense predicted probability. Shade false positives (GT=0, pred>=thresh)
    and false negatives (GT=1, pred<thresh).
    """
    target = np.asarray(r["target"]).ravel()
    pred_dense = r["pred_dense"]
    # Predictions are valid from step nrecept-1; align target slice
    start = nrecept - 1
    target_valid = target[start:]
    T = len(pred_dense)
    if len(target_valid) != T:
        target_valid = target_valid[:T]
    steps = np.arange(T) + start

    ax.fill_between(steps, 0, target_valid, color="green", alpha=0.25, label="GT label (1=disrupt)")
    ax.plot(steps, pred_dense, color="blue", linewidth=0.8, label="Pred prob")
    ax.axhline(thresh, color="gray", linestyle="--", linewidth=0.8)

    # False positives: GT=0 but pred >= thresh (model fires on clear)
    fp = (target_valid < 0.5) & (pred_dense >= thresh)
    if np.any(fp):
        ax.fill_between(steps, 0, 1, where=fp, color="red", alpha=0.35, label="FP (mistake on clear)")
    # False negatives: GT=1 but pred < thresh (model misses disruption)
    fn = (target_valid >= 0.5) & (pred_dense < thresh)
    if np.any(fn):
        ax.fill_between(steps, 0, 1, where=fn, color="orange", alpha=0.35, label="FN (missed disrupt)")

    ax.set_xlabel("Time (decimated step)")
    ax.set_ylabel("Label / Prob")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc="upper right", fontsize=6)
    ax.grid(True, alpha=0.3)
    is_clear = r["gt_step"] < 0
    title = f"Val {r['val_idx']} (seq {r['seq_idx']}) — {'CLEAR' if is_clear else 'disrupt'}"
    if is_clear and r["pred_step"] >= 0:
        title += " [FP: model fired]"
    ax.set_title(title)


n_cols = 3
n_rows = (n_vis + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)
for k, r in enumerate(results):
    i, j = k // n_cols, k % n_cols
    plot_dense_labels(r, axes[i, j], nrecept, THRESH)
for k in range(len(results), n_rows * n_cols):
    i, j = k // n_cols, k % n_cols
    axes[i, j].set_visible(False)

plt.suptitle("Dense labeling: GT (green fill) vs pred prob (blue). Red=FP (mistake on clear), Orange=FN (missed disrupt)", fontsize=10)
plt.tight_layout()
plt.savefig("evaluation_subsample_dense_labels.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Clear subsequences only: where do mistakes (false positives) lie?
clear_results = [r for r in results if r["gt_step"] < 0]
print(f"Among the {len(results)} plotted validation samples, {len(clear_results)} are CLEAR (no disruption).")
if not clear_results:
    print("No clear samples in this subset. Run with more val indices or check split.")
else:
    for r in clear_results:
        pred_dense = r["pred_dense"]
        target = np.asarray(r["target"]).ravel()
        start = nrecept - 1
        target_valid = target[start:start + len(pred_dense)]
        fp_mask = (target_valid < 0.5) & (pred_dense >= THRESH)
        n_fp = np.sum(fp_mask)
        if n_fp > 0:
            fp_steps = np.where(fp_mask)[0] + start
            print(f"  Val {r['val_idx']} (seq {r['seq_idx']}): CLEAR but model fired at {n_fp} steps — first FP at step {fp_steps[0]}, last at {fp_steps[-1]}")
        else:
            print(f"  Val {r['val_idx']} (seq {r['seq_idx']}): CLEAR, no false positives ✓")

    # Plot dense labels for clear-only samples (mistakes highlighted)
    n_clear = len(clear_results)
    n_rows_c = max(1, (n_clear + 1) // 2)
    n_cols_c = min(2, n_clear)
    fig2, axes2 = plt.subplots(n_rows_c, n_cols_c, figsize=(5 * n_cols_c, 3 * n_rows_c))
    if n_clear == 1:
        axes2 = np.array([axes2])
    else:
        axes2 = axes2.ravel()
    for k, r in enumerate(clear_results):
        plot_dense_labels(r, axes2[k], nrecept, THRESH)
    for k in range(len(clear_results), len(axes2)):
        axes2[k].set_visible(False)
    plt.suptitle("Clear subsequences: dense GT (green) vs pred (blue). Red = false positive (model wrongly fired)", fontsize=10)
    plt.tight_layout()
    plt.savefig("evaluation_subsample_clear_mistakes.png", dpi=150, bbox_inches="tight")
    plt.show()

## 6. Optional: error distribution (pred - GT) for disruptive samples

In [ ]:
errors = []
for r in results:
    if r["gt_step"] >= 0 and r["pred_step"] >= 0:
        errors.append(r["pred_step"] - r["gt_step"])

if errors:
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    ax.hist(errors, bins=min(30, len(set(errors)) + 1), edgecolor="black", alpha=0.7)
    ax.axvline(0, color="gray", linestyle="--")
    ax.set_xlabel("Predicted disruption step − GT step")
    ax.set_ylabel("Count")
    ax.set_title("Disruption timing error (validation, disruptive only)")
    plt.tight_layout()
    plt.show()
else:
    print("No disruptive samples in the plotted subset to compute errors.")